# Bank Loan Approval — EDA & Machine Learning

Complete analysis: cleaning, EDA, feature engineering, and model comparison (Logistic Regression, Decision Tree, Random Forest).

In [ ]:
import numpy as np, pandas as pd, joblib, json
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, auc)
sns.set_style('whitegrid')
df = pd.read_csv('../data/loan_data.csv')
df.head()

## 1. Dataset Overview

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe()

## 2. Missing Values

In [ ]:
df.isna().sum()

## 3. Data Cleaning

In [ ]:
for c in ['Gender','Married','Dependents','Self_Employed']:
    df[c] = df[c].fillna(df[c].mode()[0])
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])
df.isna().sum().sum()

## 4. Feature Engineering

In [ ]:
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Income_Loan_Ratio'] = df['Total_Income'] / (df['LoanAmount']+1)
df['LoanAmount_log'] = np.log1p(df['LoanAmount'])
df['Total_Income_log'] = np.log1p(df['Total_Income'])
df['Dependents'] = df['Dependents'].replace('3+','3').astype(int)
df[['Total_Income','Income_Loan_Ratio','LoanAmount_log']].head()

## 5. EDA Visualizations
### Histograms

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(12,8))
for a,col in zip(ax.ravel(),['ApplicantIncome','LoanAmount','Total_Income','Loan_Amount_Term']):
    sns.histplot(df[col].dropna(),kde=True,ax=a,color='steelblue'); a.set_title(col)
plt.tight_layout(); plt.show()

**Conclusion:** Income and loan amount are right-skewed → log transform applied.

### Boxplots (Outlier Detection)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(13,4))
for a,col in zip(ax,['ApplicantIncome','CoapplicantIncome','LoanAmount']):
    sns.boxplot(y=df[col],ax=a,color='salmon'); a.set_title(col)
plt.tight_layout(); plt.show()

**Conclusion:** Clear high-income outliers; log scaling reduces their leverage.

### Correlation Heatmap

In [ ]:
num=['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term','Credit_History','Total_Income']
plt.figure(figsize=(8,6))
sns.heatmap(df[num].corr(),annot=True,cmap='coolwarm',fmt='.2f',square=True)
plt.title('Correlation Heatmap'); plt.show()

**Conclusion:** Credit history is the strongest signal for approval.

## 6. Encoding & Model Training

In [ ]:
enc = {'Gender':{'Male':1,'Female':0},'Married':{'Yes':1,'No':0},
       'Education':{'Graduate':1,'Not Graduate':0},'Self_Employed':{'Yes':1,'No':0},
       'Property_Area':{'Urban':2,'Semiurban':1,'Rural':0},'Loan_Status':{'Y':1,'N':0}}
for col,m in enc.items(): df[col]=df[col].map(m)
features = ['Gender','Married','Dependents','Education','Self_Employed','Credit_History',
    'Property_Area','Total_Income_log','LoanAmount_log','Income_Loan_Ratio']
X=df[features]; y=df['Loan_Status']
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
scaler=StandardScaler().fit(X_tr)
X_tr_s,X_te_s = scaler.transform(X_tr),scaler.transform(X_te)

In [ ]:
models={'Logistic Regression':(LogisticRegression(max_iter=1000,random_state=42),True),
 'Decision Tree':(DecisionTreeClassifier(max_depth=5,random_state=42),False),
 'Random Forest':(RandomForestClassifier(n_estimators=200,max_depth=8,random_state=42),False)}
results={}
for name,(model,scale) in models.items():
    Xtr,Xte=(X_tr_s,X_te_s) if scale else (X_tr,X_te)
    model.fit(Xtr,y_tr); pred=model.predict(Xte)
    results[name]={'Accuracy':accuracy_score(y_te,pred),'Precision':precision_score(y_te,pred),
        'Recall':recall_score(y_te,pred),'F1':f1_score(y_te,pred)}
pd.DataFrame(results).T.round(4)

## 7. Confusion Matrices & ROC

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,(name,(model,scale)) in zip(axes,models.items()):
    Xte=X_te_s if scale else X_te
    cm=confusion_matrix(y_te,model.predict(Xte))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,cbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 8. Conclusion
Logistic Regression achieves the best accuracy and F1, and is the most interpretable — recommended for production loan decisions.